# SigExt Training (25k Samples)

## Semantic & Multilingual Salient Information Prompting

This notebook trains the **SigExt (Salient Information Extractor)** model on **25,000 samples** from the WITS dataset using semantic supervision.

### Comparison with 10k Model
This variant uses **2.5x more training data** to study the effect of dataset size on:
- Salience detection accuracy
- Summary quality (ROUGE, BERTScore)
- Keyphrase inclusion rate (KIR)

### Pipeline Overview
1. **Data Preparation**: Generate semantic labels using Sentence-BERT
2. **Model Training**: Fine-tune XLM-RoBERTa Longformer for token classification
3. **Upload**: Push trained model to Hugging Face Hub

### Configuration
- **Base Model**: `markussagen/xlm-roberta-longformer-base-4096`
- **SBERT Model**: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`
- **Training Samples**: 25,000
- **Semantic Threshold**: 0.60

---
## 1. Environment Setup

Install all required dependencies for training.

In [4]:
%%capture
!pip install transformers datasets accelerate bitsandbytes sentence-transformers \
    spacy rouge_score bert_score langchain langchain-community langchain-huggingface \
    huggingface_hub "numpy<2.0" "scipy>=1.10"
!python -m spacy download it_core_news_sm

In [5]:
import os
import json
import gc
import shutil
import torch
import torch.nn as nn
import spacy
import numpy as np
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    Trainer, 
    TrainingArguments
)
from huggingface_hub import login, HfApi

---
## 2. Configuration

Define all hyperparameters and model paths.

> **Note**: The only difference from the 10k notebook is `NUM_TRAIN_SAMPLES` and `REPO_NAME`.

In [ ]:
# Hugging Face authentication
# Option 1: Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except:
    # Option 2: Environment variable or manual input
    HF_TOKEN = os.getenv("HF_TOKEN") or "YOUR_HF_TOKEN_HERE"

# Repository configuration
HF_USERNAME = "LookUpMark"  # Change to your username
REPO_NAME = "sigext-wits-it-25k-060t"  # Different from 10k!

# Training configuration
CONFIG = {
    # Models
    "SBERT_MODEL": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    "LONGFORMER_MODEL": "markussagen/xlm-roberta-longformer-base-4096",
    
    # Training parameters
    "MAX_LEN": 2048,
    "SEMANTIC_THRESHOLD": 0.60,
    "NUM_TRAIN_SAMPLES": 25000,  # 2.5x more than 10k variant
    
    # File paths
    "TRAIN_FILE": "wits_train_25k_060t.jsonl",
    "OUTPUT_DIR": "./sigext_25k_060t_final",
    
    # Data filtering
    "MIN_SOURCE_LEN": 500,
    "MAX_SOURCE_LEN": 10000,
    "MIN_SUMMARY_LEN": 50,
    "MIN_SENT_LEN": 20,
    "BATCH_SIZE": 32
}

# Authenticate with Hugging Face
login(token=HF_TOKEN)

---
## 3. Semantic Labeling

### Why Semantic Labeling?
The original SIP paper uses fuzzy (lexical) matching to create training labels. This fails for:
- Synonyms: "car" vs "vehicle"
- Paraphrases: "The company failed" vs "bankruptcy was declared"

### Our Approach
We use **Sentence-BERT** embeddings to compute semantic similarity between:
- Each source sentence
- All summary sentences

A source sentence is labeled as **salient (1)** if its maximum similarity to any summary sentence exceeds the threshold (0.60).

In [7]:
def prepare_training_data():
    """
    Generate semantic labels for training data.
    
    Process:
    1. Load WITS dataset (Italian Wikipedia summaries)
    2. For each document, segment into sentences
    3. Compute SBERT embeddings for all sentences
    4. Label sentences based on semantic similarity to summary
    5. Save to JSONL file
    """
    # Load Italian Spacy model for sentence segmentation
    nlp = spacy.load("it_core_news_sm")

    # Load Sentence-BERT (multilingual)
    print("=" * 60)
    print("Loading Sentence-BERT model...")
    print("=" * 60)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    sbert = SentenceTransformer(CONFIG["SBERT_MODEL"], device=device)
    
    # Load WITS dataset in streaming mode (memory efficient)
    print("\n" + "=" * 60)
    print("Loading WITS dataset...")
    print("=" * 60)
    dataset = load_dataset("silvia-casola/WITS", split="train", streaming=True)
    
    # Process and label documents
    print("\n" + "=" * 60)
    print("Generating semantic labels...")
    print("=" * 60)
    
    count = 0
    with open(CONFIG["TRAIN_FILE"], "w") as f_out:
        pbar = tqdm(total=CONFIG["NUM_TRAIN_SAMPLES"], desc="Labeling")
        
        for entry in dataset:
            source = entry['source']
            summary = entry['summary']
            
            # Quality filters
            if (len(source) < CONFIG["MIN_SOURCE_LEN"] or 
                len(summary) < CONFIG["MIN_SUMMARY_LEN"] or 
                len(source) > CONFIG["MAX_SOURCE_LEN"]):
                continue

            # Segment into sentences
            doc_sents = [s.text for s in nlp(source).sents 
                        if len(s.text) > CONFIG["MIN_SENT_LEN"]]
            sum_sents = [s.text for s in nlp(summary).sents 
                        if len(s.text) > CONFIG["MIN_SENT_LEN"]]

            if not doc_sents or not sum_sents:
                continue

            # Compute SBERT embeddings
            doc_emb = sbert.encode(
                doc_sents, convert_to_tensor=True, 
                show_progress_bar=False, batch_size=CONFIG["BATCH_SIZE"]
            )
            sum_emb = sbert.encode(
                sum_sents, convert_to_tensor=True, 
                show_progress_bar=False, batch_size=CONFIG["BATCH_SIZE"]
            )
            
            # Compute similarity matrix and generate labels
            scores = util.cos_sim(doc_emb, sum_emb)
            labels = [
                1 if scores[i].max().item() > CONFIG["SEMANTIC_THRESHOLD"] else 0
                for i in range(len(doc_sents))
            ]
            
            # Only keep documents with at least one salient sentence
            if 1 in labels:
                f_out.write(json.dumps({
                    "sentences": doc_sents, 
                    "labels": labels
                }) + "\n")
                count += 1
                pbar.update(1)
            
            if count >= CONFIG["NUM_TRAIN_SAMPLES"]:
                break
        
        pbar.close()

    print(f"\nLabeling completed! Saved {count} documents to {CONFIG['TRAIN_FILE']}")
    
    # Cleanup GPU memory
    del sbert
    torch.cuda.empty_cache()
    gc.collect()

In [8]:
# Generate training data
prepare_training_data()

Loading Sentence-BERT model...


Repo card metadata block was not found. Setting CardData to empty.



Loading WITS dataset...

Generating semantic labels...


Labeling:   0%|          | 0/25000 [00:00<?, ?it/s]


Labeling completed! Saved 25000 documents to wits_train_25k_060t.jsonl


---
## 4. Dataset & Trainer Classes

### SigExtDataset
PyTorch Dataset that:
- Loads pre-labeled JSONL data
- Tokenizes with Longformer tokenizer
- Aligns sentence-level labels to token-level

### WeightedTrainer
Custom HuggingFace Trainer that:
- Uses weighted cross-entropy loss
- Addresses class imbalance (only ~10% sentences are salient)
- Weight ratio: 1:10 (non-salient:salient)

In [9]:
class SigExtDataset(Dataset):
    """
    PyTorch Dataset for SigExt training.
    
    Handles tokenization and label alignment for Longformer.
    Labels are assigned at sentence-level but model outputs per-token.
    """
    
    def __init__(self, path: str, tokenizer):
        self.data = [json.loads(line) for line in open(path)]
        self.tokenizer = tokenizer
        print(f"Loaded {len(self.data)} training examples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = " ".join(item['sentences'])
        labels = item['labels']

        # Tokenize
        enc = self.tokenizer(
            text, 
            truncation=True, 
            max_length=CONFIG["MAX_LEN"], 
            padding="max_length"
        )

        # Create token-level labels
        # -100 = ignore index (for padding and non-first tokens)
        token_labels = [-100] * len(enc['input_ids'])
        limit = min(len(labels), CONFIG["MAX_LEN"])
        token_labels[:limit] = labels[:limit]

        return {
            "input_ids": torch.tensor(enc['input_ids']),
            "attention_mask": torch.tensor(enc['attention_mask']),
            "labels": torch.tensor(token_labels)
        }


class WeightedTrainer(Trainer):
    """
    Custom Trainer with weighted loss for class imbalance.
    
    The positive class (salient) is weighted 10x higher than
    the negative class to prevent the model from predicting
    all zeros (which would give ~90% accuracy but 0% recall).
    """
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        device = inputs["input_ids"].device

        # Weighted cross-entropy: [non-salient=1.0, salient=10.0]
        class_weights = torch.tensor([1.0, 10.0]).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
        
        loss = criterion(
            outputs.get("logits").view(-1, 2),
            labels.view(-1)
        )
        
        return (loss, outputs) if return_outputs else loss

---
## 5. Model Training

### Training Configuration
- **Epochs**: 2
- **Batch Size**: 2 (with gradient accumulation = 4, effective batch = 8)
- **Learning Rate**: 2e-5
- **Precision**: FP16 (mixed precision for memory efficiency)

### Expected Training Time
With 25k samples, expect ~2.5x longer training time compared to 10k:
- 10k: ~3 hours on T4 GPU
- 25k: ~7-8 hours on T4 GPU

In [10]:
def train_sigext():
    """
    Train the SigExt model on prepared data.
    """
    print("=" * 60)
    print("Loading Longformer model...")
    print("=" * 60)
    
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    model = AutoModelForTokenClassification.from_pretrained(
        CONFIG["LONGFORMER_MODEL"], 
        num_labels=2  # Binary: salient vs non-salient
    )

    # Training arguments optimized for T4 GPU
    args = TrainingArguments(
        output_dir="./checkpoints",
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        fp16=True,
        save_strategy="epoch",
        logging_steps=100,
        report_to="none"
    )
    
    # Create dataset and trainer
    dataset = SigExtDataset(CONFIG["TRAIN_FILE"], tokenizer)
    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=dataset
    )
    
    # Train
    print("\n" + "=" * 60)
    print("Starting training...")
    print("=" * 60)
    trainer.train()
    
    # Save final model
    print("\n" + "=" * 60)
    print("Saving model...")
    print("=" * 60)
    model.save_pretrained(CONFIG["OUTPUT_DIR"])
    tokenizer.save_pretrained(CONFIG["OUTPUT_DIR"])

    print(f"\nTraining completed! Model saved to {CONFIG['OUTPUT_DIR']}")
    
    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

In [11]:
# Train the model
train_sigext()

Loading Longformer model...


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at markussagen/xlm-roberta-longformer-base-4096 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded 25000 training examples

Starting training...


Step,Training Loss
100,0.621100
200,0.605300
300,0.606100
400,0.587200
500,0.590600
600,0.588500
700,0.603800
800,0.592600
900,0.590900
1000,0.597300



Saving model...

Training completed! Model saved to ./sigext_25k_060t_final


---
## 6. Upload to Hugging Face Hub

Push the trained model to Hugging Face for easy sharing and inference.

In [12]:
def upload_to_hub():
    """
    Upload trained model to Hugging Face Hub.
    """
    repo_id = f"{HF_USERNAME}/{REPO_NAME}"
    api = HfApi()
    
    try:
        # Create repository (or get existing)
        api.create_repo(repo_id=repo_id, exist_ok=True)
        print(f"Repository '{repo_id}' ready.")
    except Exception as e:
        print(f"Error creating repository: {e}")
        return

    try:
        # Upload model files
        api.upload_folder(
            folder_path=CONFIG["OUTPUT_DIR"],
            repo_id=repo_id,
            repo_type="model",
            commit_message="Training completed with 25k samples"
        )
        print(f"\nModel successfully uploaded to: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"Error uploading model: {e}")

In [13]:
# Upload to Hugging Face
upload_to_hub()

Repository 'LookUpMark/sigext-wits-it-25k-060t' ready.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Model successfully uploaded to: https://huggingface.co/LookUpMark/sigext-wits-it-25k-060t


---
## Summary

This notebook completed the following steps:

1. **Data Preparation**: Generated 25,000 semantically labeled training examples
2. **Model Training**: Fine-tuned XLM-RoBERTa Longformer for salience detection
3. **Hub Upload**: Published model to Hugging Face for inference

### Ablation Study: 10k vs 25k

| Metric | 10k Model | 25k Model |
|--------|-----------|------------|
| BERTScore | ___ | ___ |
| ROUGE-1 | ___ | ___ |
| KIR | ___ | ___ |

### Expected Observations
- **Higher recall**: More training data helps detect more salient sentences
- **Better generalization**: Larger dataset reduces overfitting
- **Diminishing returns**: Improvement from 10k→25k is less than 0→10k